# HiveMind Benchmark Autopilot (3 Suites, Foreground + Live Google Drive Sync)

Dieses Notebook führt **alle 3 HiveMind Benchmarks** nacheinander im **Vordergrund** aus, sodass Google Colab die laufende Zelle sieht und **nicht durch `tmux`-Leerlauf in den Idle-Disconnect geht**.

### Was dieses Notebook automatisch erledigt:
1. **Vorher-Sync mit Google Drive:** Mountet Drive gleich am Anfang und stellt vorhandene Checkpoints (`MyDrive/hivemind_bench/results/*.json`) wieder her.
2. **Live-Zwischenstand-Push nach jeder einzelnen Aufgabe:** Nach jeder Task × Harness-Zelle schreibt `runner.py` atomar lokal **und** spiegelt den aktuellen JSON-Stand sofort nach Google Drive (`--sync-dir`).
3. **Nahtloses Auto-Resume (`--resume`):** Falls Colab nach vielen Stunden trennt oder neu startet, einfach Zelle 1 → 2 → 3 erneut ausführen: bereits erledigte `(run, task, condition)`-Zellen werden aus Drive geladen, automatisch übersprungen, und der Lauf macht exakt dort weiter, wo er aufgehört hat.
4. **Identische Fairness-Regeln (`2722e92`):** Gleiches Modell, gleicher `llama-server`, **identisches Timeout** über alle Harnesses (`300s` Core/Polyglot, `1800s` DeepSWE), inkl. echtem Token- & Cache-Tracking für HiveMind, Aider, Hermes (`state.db`) und Pi (`--mode json`).

### Die 3 Benchmark-Phasen im Autopilot:
1. **Core Suite (`23 Tasks`)** — Standard: **4B-Modell, 3 Runs**, alle Harnesses (HiveMind `duo_core`, Aider, Hermes, Pi).
2. **Polyglot Python (`34 Tasks`)** — Standard: **4B-Modell, 1 Run** (einstellbar), alle Harnesses. Optional danach 35B.
3. **DeepSWE Real-World (`3 Repo-Tasks @ 128k`)** — optional, nur A100 + 35B; standardmäßig aus, weil es lange dauert.

**Toolset:** `hivemind_duo` nutzt `duo_core` (18 Tools: read/edit/write/run_bash/run_tests/task_complete/...). `hivemind_duo_full` ist optional und nutzt das volle Toolset.


**Token-Tracking und Timeout:** HiveMind zählt `usage_meta` (inkl. Cache), Aider parst die gerundete CLI-Zeile, Hermes liest `state.db`, Pi liest `--mode json`. Die Zahlen sind deshalb **nicht exakt dieselbe Quelle**; Pass-Rate und Zeit sind der primäre Vergleich, Token-Zahlen nur mit Vorsicht. Alle Harnesses nutzen jetzt denselben `_agent_timeout_s` (Task-Timeout bzw. `HM_CELL_TIMEOUT`, je nachdem was kleiner ist).


In [ ]:
# 1) Google Drive mounten & vorhandene Checkpoints prüfen
import json
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hivemind_bench")
DRIVE_RESULTS = DRIVE_ROOT / "results"
DRIVE_LOGS = DRIVE_ROOT / "logs"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    DRIVE_LOGS.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive verbunden: {DRIVE_RESULTS}")
except ImportError:
    DRIVE_RESULTS = Path("/root/HiveMind_llama/bench/results_backup")
    DRIVE_LOGS = Path("/root/HiveMind_llama/bench/logs_backup")
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    DRIVE_LOGS.mkdir(parents=True, exist_ok=True)
    print(f"Hinweis: google.colab nicht aktiv, lokaler Backup-Ordner: {DRIVE_RESULTS}")

existing = sorted(DRIVE_RESULTS.glob("*.json"))
if existing:
    print(f"\nGefundene Checkpoints in Drive ({len(existing)}):")
    for p in existing:
        try:
            d = json.loads(p.read_text(encoding="utf-8"))
            print(f"  - {p.name:30s} | model={d.get('model')} ctx={d.get('num_ctx', '?')} rows={len(d.get('results', []))}")
        except Exception as exc:
            print(f"  - {p.name:30s} | (nicht lesbar: {exc})")
else:
    print("\nNoch keine Checkpoints in Drive — frischer Start.")

In [ ]:
# 2) Repo aktualisieren, Drive -> Lokal vorab syncen & Basis-Setup (inkl. Polyglot & DeepSWE)
import shutil, subprocess, sys
from pathlib import Path

if "DRIVE_RESULTS" not in globals():
    DRIVE_ROOT = Path("/content/drive/MyDrive/hivemind_bench")
    DRIVE_RESULTS = DRIVE_ROOT / "results" if Path("/content/drive/MyDrive").exists() else Path("/root/HiveMind_llama/bench/results_backup")
    DRIVE_LOGS = DRIVE_ROOT / "logs" if Path("/content/drive/MyDrive").exists() else Path("/root/HiveMind_llama/bench/logs_backup")
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    DRIVE_LOGS.mkdir(parents=True, exist_ok=True)

def _run_stream(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)
    assert p.stdout is not None
    for line in p.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed (exit={rc}): {cmd}")

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!df -h /root | tail -1

![ -d /root/HiveMind_llama/.git ] || git clone --branch bench-stable https://github.com/BoredLuzo/HiveMind_llama.git /root/HiveMind_llama
!cd /root/HiveMind_llama && git fetch origin bench-stable && git checkout bench-stable && git reset --hard origin/bench-stable
!ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh 2>/dev/null || cp -f /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!cd /root/HiveMind_llama && git log -1 --format='[bench] commit: %h %s'

# Vorher-Sync: Alle Drive-Checkpoints in das lokale results/-Verzeichnis ziehen
local_res = Path("/root/HiveMind_llama/bench/results")
local_res.mkdir(parents=True, exist_ok=True)
for src in sorted(DRIVE_RESULTS.glob("*.json")):
    dst = local_res / src.name
    if (not dst.exists()) or src.stat().st_size >= dst.stat().st_size:
        shutil.copy2(src, dst)
        print(f"[pre-sync] {src.name} -> {dst}")

# Basis-Setup (Python 3.14, llama.cpp, Aider, Hermes, Pi + Node 22)
!HM_MODEL="skip" HM_LLAMA_CUDA=12.8 bash /root/rent_bootstrap.sh setup

# Alle 3 Task-Corpora bereithalten (Core ist im Repo, Polyglot & DeepSWE werden importiert)
_poly_dir = Path("/root/HiveMind_llama/bench/tasks_polyglot")
_poly_count = len(list(_poly_dir.glob("polyglot_*/task.json")))
if _poly_count < 34:
    print(f"[setup] Polyglot nur {_poly_count}/34 Tasks vorhanden — importiere vollständigen Corpus.")
    _run_stream(["bash", "-lc", "HM_POLYGLOT_NAMES=all bash /root/rent_bootstrap.sh polyglot"])
else:
    print("[setup] Polyglot Tasks (34) bereits vollständig.")

_run_stream(["pip", "install", "-q", "hypothesis", "trio", "attrs", "pytest", "pytest-asyncio", "anyio"])
if not Path("/root/HiveMind_llama/bench/tasks_deepswe/SOURCE.json").is_file():
    _run_stream(["bash", "/root/rent_bootstrap.sh", "deepswe"])
else:
    print("[setup] DeepSWE Tasks (3) bereits vorhanden.")


In [ ]:
# 3) AUTOPILOT: 4B Core (3 Runs) -> 4B Polyglot (1 Run), Live-Drive-Sync
import os
import subprocess as _subprocess
import sys as _sys
from pathlib import Path

if "DRIVE_RESULTS" not in globals() or "DRIVE_LOGS" not in globals():
    _dr = Path("/content/drive/MyDrive/hivemind_bench")
    DRIVE_RESULTS = (_dr / "results") if Path("/content/drive/MyDrive").exists() else Path("/root/HiveMind_llama/bench/results_backup")
    DRIVE_LOGS = (_dr / "logs") if Path("/content/drive/MyDrive").exists() else Path("/root/HiveMind_llama/bench/logs_backup")
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    DRIVE_LOGS.mkdir(parents=True, exist_ok=True)

# Standard: alle Harnesses. Optional: "hivemind_duo,hivemind_duo_full,aider,hermes,pi"
CONDITIONS = "hivemind_duo,aider,hermes,pi"
CTX_35B_SMALL_SUITES = 32768
CORE_TASKS = "bench/tasks"
POLYGLOT_TASKS = "bench/tasks_polyglot"
DEEPSWE_TASKS = "bench/tasks_deepswe"


def _detect_vram_gb():
    try:
        out = _subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
            text=True,
        ).strip().splitlines()[0]
        return float(out) / 1024.0
    except Exception:
        return 15.0


def run_autopilot(model_choice="4b", core_runs=3, polyglot_runs=1,
                  deepswe_runs=0, conditions=CONDITIONS):
    vram_gb = _detect_vram_gb()
    use_model = str(model_choice).strip().lower()
    if use_model == "auto":
        use_model = "4b" if vram_gb < 20.0 else "35b"

    if use_model == "4b":
        model_id = "qwen3.5:4b-mtp"
        tag = "4b"
        # 128k KV on a 16 GB T4 is not guaranteed; safe 32k unless L4/A100.
        ctx_small = 131072 if vram_gb >= 20.0 else 32768
        ctx_deepswe = 131072
    else:
        model_id = "qwen3.6:35b-a3b-ud"
        tag = "35b"
        ctx_small = CTX_35B_SMALL_SUITES
        ctx_deepswe = 131072

    if deepswe_runs and vram_gb < 35.0:
        print(f"[autopilot] SKIP DeepSWE: {vram_gb:.1f} GB VRAM < 35 GB (A100 required).")
        deepswe_runs = 0

    stages = []
    if core_runs:
        stages.append({"name": f"core_{tag}_{ctx_small // 1024}k",
                       "tasks": CORE_TASKS, "model": model_id,
                       "ctx": ctx_small, "runs": int(core_runs),
                       "cell_timeout": 900})
    if polyglot_runs:
        stages.append({"name": f"polyglot_{tag}_{ctx_small // 1024}k",
                       "tasks": POLYGLOT_TASKS, "model": model_id,
                       "ctx": ctx_small, "runs": int(polyglot_runs),
                       "cell_timeout": 900})
    if deepswe_runs:
        stages.append({"name": f"deepswe_{tag}_{ctx_deepswe // 1024}k",
                       "tasks": DEEPSWE_TASKS, "model": model_id,
                       "ctx": ctx_deepswe, "runs": int(deepswe_runs),
                       "cell_timeout": 1800})

    print(f"=== AUTOPILOT (GPU {vram_gb:.1f} GB -> {use_model.upper()} / {model_id}) ===")
    print(f"Conditions: {conditions}")
    print("HiveMind toolset: duo_core (18 tools)" if "hivemind_duo" in conditions else "HiveMind: off")
    print(f"Drive Sync: {DRIVE_RESULTS}")
    print(f"Stages: {[s['name'] for s in stages]}\n")

    for idx, st in enumerate(stages, 1):
        out_file = Path(f"/root/HiveMind_llama/bench/results/{st['name']}.json")
        log_file = DRIVE_LOGS / f"{st['name']}.log"
        print(f"\n=================================================================")
        print(f"[STAGE {idx}/{len(stages)}] {st['name']} | tasks={st['tasks']} | runs={st['runs']} | ctx={st['ctx']}")
        print(f"=================================================================", flush=True)
        env = os.environ.copy()
        env.update({
            "HM_MODEL": st["model"],
            "HM_CTX": str(st["ctx"]),
            "HM_TASKS": st["tasks"],
            "HM_RUNS": str(st["runs"]),
            "HM_RUN_OFFSET": "0",
            "HM_CONDITIONS": conditions,
            "HM_CELL_TIMEOUT": str(st["cell_timeout"]),
            "HM_OUT": str(out_file),
            "HM_SYNC_DIR": str(DRIVE_RESULTS),
            "HM_RESUME": "1",
        })
        with open(log_file, "a", encoding="utf-8") as lf:
            lf.write(f"\n=== STAGE {st['name']} START ===\n")
            proc = _subprocess.Popen(
                ["bash", "/root/rent_bootstrap.sh", "fg"], env=env,
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, encoding="utf-8", errors="replace", bufsize=1,
            )
            try:
                assert proc.stdout is not None
                for line in proc.stdout:
                    _sys.stdout.write(line)
                    _sys.stdout.flush()
                    lf.write(line)
                    lf.flush()
                rc = proc.wait()
            except KeyboardInterrupt:
                proc.terminate()
                proc.wait(timeout=10)
                raise
            if rc != 0:
                raise RuntimeError(f"Stage {st['name']} abgebrochen (exit={rc}). Zwischenstand: {DRIVE_RESULTS}")

    print(f"\nALLE {len(stages)} STAGES ABGESCHLOSSEN. Ergebnisse: {DRIVE_RESULTS}")


# Standardlauf: 4B, Core 3 Runs, dann Polyglot 1 Run.
run_autopilot("4b", core_runs=3, polyglot_runs=1, deepswe_runs=0)


In [ ]:
# 3b) OPTIONAL: dieselben Suiten mit 35B (A100/L4)
# Nach Zelle 3 ausführen; die 4B-Ergebnisse bleiben getrennt (eigene Dateinamen).
run_autopilot("35b", core_runs=1, polyglot_runs=1, deepswe_runs=0)


In [ ]:
# 3c) OPTIONAL: DeepSWE (nur A100 + 35B, 128k, 1 Run) - kann sehr lange dauern
run_autopilot("35b", core_runs=0, polyglot_runs=0, deepswe_runs=1)


In [ ]:
# 4) Leaderboard über alle in Google Drive gespeicherten Benchmarks
import glob, subprocess as _subprocess_lb
from pathlib import Path

if "DRIVE_RESULTS" not in globals():
    DRIVE_RESULTS = Path("/content/drive/MyDrive/hivemind_bench/results") if Path("/content/drive/MyDrive").exists() else Path("/root/HiveMind_llama/bench/results")

files = sorted(glob.glob(str(DRIVE_RESULTS / "*.json")))
if not files:
    print("Noch keine JSON-Ergebnisse gefunden.")
else:
    for f in files:
        print(f"\n==================== {Path(f).name} ====================")
        _subprocess_lb.run(
            ["/root/HiveMind_llama/.venv/bin/python", "/root/HiveMind_llama/bench/analyze.py", "--results", f, "--by-model"],
            check=False,
        )


In [ ]:
# 5) Server & VRAM sauber freigeben
!bash /root/rent_bootstrap.sh stop
